In [1]:
import pandas as pd
import numpy as np
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
print("Libraries loaded!")

Libraries loaded!


In [6]:
# Cell 2: Load and explore the dataset
df = pd.read_csv('C:/Users/harsh/Projects/sql-supply-chain/data/DataCoSupplyChainDataset.csv', 
                 encoding='latin-1')
# encoding='latin-1' — this dataset has special characters
# without this you'll get a UnicodeDecodeError

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (180519, 53)

Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Sta

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [7]:
# Cell 3: Clean the data

# Drop columns that are mostly empty or useless for analysis
df.drop(columns=['Product Description', 'Order Zipcode'], inplace=True)

# Check remaining shape
print(f"Shape after dropping: {df.shape}")

# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Extract month and year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month_name'] = df['order date (DateOrders)'].dt.month_name()

# Check for remaining missing values
print(f"\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData types of key columns:")
print(df[['order date (DateOrders)', 'Benefit per order', 'Sales', 'Order Item Quantity']].dtypes)

Shape after dropping: (180519, 51)

Remaining missing values:
Customer Lname      8
Customer Zipcode    3
dtype: int64

Data types of key columns:
order date (DateOrders)    datetime64[us]
Benefit per order                 float64
Sales                             float64
Order Item Quantity                 int64
dtype: object


In [8]:
conn = sqlite3.connect(':memory:')
df.to_sql('supply_chain', conn, index=False, if_exists='replace')
print("SQL loaded!")
print(f"Total rows: {pd.read_sql_query('SELECT COUNT(*) as total FROM supply_chain', conn)['total'][0]:,}")

SQL loaded!
Total rows: 180,519


In [14]:
query1 = """
select
    count(*) as total_orders,
    round(sum(sales),2) as total_sales,
    round(sum(`benefit per order`),2) as total_benefit,
    round(avg(`benefit per order`),2) as average_benfit_per_order
from supply_chain
"""
result1=pd.read_sql_query(query1,conn)
print("\n=== Overall Business Summary ===")
print(result1)




=== Overall Business Summary ===
   total_orders  total_sales  total_benefit  average_benfit_per_order
0        180519  36784735.01     3966902.97                     21.97


In [15]:
print([col for col in df.columns if 'enefit' in col or 'ales' in col or 'ategory' in col])

['Benefit per order', 'Sales per customer', 'Category Id', 'Category Name', 'Sales', 'Product Category Id']


In [18]:
query2 = """
select
    `category name`,
    round(sum(sales),2) as total_sales,
    round(sum(`benefit per order`),2) as total_benefit,
    round(avg(`benefit per order`),2) as average_benfit_per_order,
    count(*) as total_orders
from supply_chain
group by `category name`
order by total_benefit desc
"""
result2=pd.read_sql_query(query2,conn)
print("\n=== Overall performance ===")
print(result2)


=== Overall performance ===
           Category Name  total_sales  total_benefit  \
0                Fishing   6929653.69      756220.77   
1                 Cleats   4431942.78      494636.92   
2       Camping & Hiking   4118425.57      427455.57   
3       Cardio Equipment   3694843.20      383011.10   
4        Women's Apparel   3147800.00      350421.03   
5           Water Sports   3113844.68      325146.96   
6   Indoor/Outdoor Games   2888993.91      318451.43   
7         Men's Footwear   2891757.66      311902.82   
8          Shop By Sport   1309522.04      129813.96   
9              Computers    663000.00       69656.81   
10           Electronics    371034.64       40891.38   
11                Garden    257768.73       33443.01   
12              Cameras     267607.69       30289.80   
13   Children's Clothing    232829.20       27178.10   
14                Crafts    223356.33       25531.17   
15      Women's Clothing    140283.00       19102.85   
16        Girls' Ap

In [23]:
query3 = """
select
    `category name`,
    `order status`,
    `shipping mode`,
    round(sales,2),
    round(`benefit per order`,2) as benefit
from supply_chain
where `benefit per order` < 0
order by `benefit per order` asc
limit 20
"""
result3=pd.read_sql_query(query3,conn)
print("\n=== Overall performance ===")
print(result3)


=== Overall performance ===
        Category Name     Order Status   Shipping Mode  round(sales,2)  \
0   Strength Training         COMPLETE    Second Class         1999.99   
1           Computers       PROCESSING    Second Class         1500.00   
2           Computers           CLOSED  Standard Class         1500.00   
3           Computers  PENDING_PAYMENT  Standard Class         1500.00   
4           Computers           CLOSED  Standard Class         1500.00   
5           Computers         COMPLETE  Standard Class         1500.00   
6           Computers          PENDING  Standard Class         1500.00   
7           Computers       PROCESSING  Standard Class         1500.00   
8           Computers          ON_HOLD    Second Class         1500.00   
9           Computers         COMPLETE  Standard Class         1500.00   
10          Computers         COMPLETE     First Class         1500.00   
11          Computers  PENDING_PAYMENT  Standard Class         1500.00   
12       

In [28]:
print([col for col in df.columns if 'late_deliver' in col.lower() or 'ship' in col.lower() or 'deliver' in col.lower()])

['Days for shipping (real)', 'Days for shipment (scheduled)', 'Delivery Status', 'Late_delivery_risk', 'shipping date (DateOrders)', 'Shipping Mode']


In [31]:
query4 = """
SELECT
    shipping_mode,
    total_orders,
    late_deliveries,
    ROUND(late_deliveries * 100.0 / total_orders, 1) as late_pct,
    avg_benefit
FROM (
    SELECT
        `Shipping Mode` as shipping_mode,
        COUNT(*) as total_orders,
        SUM(CASE WHEN `Late_delivery_risk` = 1 THEN 1 ELSE 0 END) as late_deliveries,
        ROUND(AVG(`Benefit per order`), 2) as avg_benefit
    FROM supply_chain
    GROUP BY `Shipping Mode`
) sub
ORDER BY late_pct DESC
"""
result4=pd.read_sql_query(query4,conn)
print("\n=== Overall performance ===")
print(result4)


=== Overall performance ===
    shipping_mode  total_orders  late_deliveries  late_pct  avg_benefit
0     First Class         27814            26513      95.3        23.12
1    Second Class         35216            26987      76.6        21.31
2        Same Day          9737             4454      45.7        20.85
3  Standard Class        107752            41023      38.1        22.00
